# Hugging Face Transformers 微调训练入门

本示例将介绍基于 Transformers 实现模型微调训练的主要流程，包括：
- 数据集下载
- 数据预处理
- 训练超参数配置
- 训练评估指标设置
- 训练器基本介绍
- 实战训练
- 模型保存

## YelpReviewFull 数据集

**Hugging Face 数据集：[ YelpReviewFull ](https://huggingface.co/datasets/yelp_review_full)**

### 数据集摘要

Yelp评论数据集包括来自Yelp的评论。它是从Yelp Dataset Challenge 2015数据中提取的。

### 支持的任务和排行榜
文本分类、情感分类：该数据集主要用于文本分类：给定文本，预测情感。

### 语言
这些评论主要以英语编写。

### 数据集结构

#### 数据实例
一个典型的数据点包括文本和相应的标签。

来自YelpReviewFull测试集的示例如下：

```json
{
    'label': 0,
    'text': 'I got \'new\' tires from them and within two weeks got a flat. I took my car to a local mechanic to see if i could get the hole patched, but they said the reason I had a flat was because the previous patch had blown - WAIT, WHAT? I just got the tire and never needed to have it patched? This was supposed to be a new tire. \\nI took the tire over to Flynn\'s and they told me that someone punctured my tire, then tried to patch it. So there are resentful tire slashers? I find that very unlikely. After arguing with the guy and telling him that his logic was far fetched he said he\'d give me a new tire \\"this time\\". \\nI will never go back to Flynn\'s b/c of the way this guy treated me and the simple fact that they gave me a used tire!'
}
```

#### 数据字段

- 'text': 评论文本使用双引号（"）转义，任何内部双引号都通过2个双引号（""）转义。换行符使用反斜杠后跟一个 "n" 字符转义，即 "\n"。
- 'label': 对应于评论的分数（介于1和5之间）。

#### 数据拆分

Yelp评论完整星级数据集是通过随机选取每个1到5星评论的130,000个训练样本和10,000个测试样本构建的。总共有650,000个训练样本和50,000个测试样本。

## 下载数据集

In [3]:
from datasets import load_dataset

dataset = load_dataset("yelp_review_full")

In [4]:
dataset

DatasetDict({
    train: Dataset({
        features: ['label', 'text'],
        num_rows: 650000
    })
    test: Dataset({
        features: ['label', 'text'],
        num_rows: 50000
    })
})

In [5]:
dataset["train"][118]

{'label': 0,
 'text': "This is the absolute WORST Steak N Shake I've ever been to. \\n\\nThe bf and I got lost around Pittsburgh for 40 min. trying to find this location and on top of the unnecessary 1 hour wait for our food, the food itself was just so crappily made. It was so bad and we had waited a ridiculous amount of time (c'mon, Steak N Shake is basically a glorified McD's), our waiter comped our WHOLE MEAL.\\n\\nIf not for the compensation, I would have been pissed that we had to pay for such a crappy meal. The bf, on the other hand, was more upset about the principle of the matter... we had trekked all that way just to get our Steak N Shake fix (since both of us haven't had it since our undergraduate days) and it was a major fail.\\n\\nI think I'm good with my Steak N Shake craving for a long time now...\\n\\n\\n\\nPS. The whole joint smells like major B.O."}

In [6]:
import random
import pandas as pd
import datasets
from IPython.display import display, HTML

In [7]:
def show_random_elements(dataset, num_examples=10):
    assert num_examples <= len(dataset), "Can't pick more elements than there are in the dataset."
    picks = []
    for _ in range(num_examples):
        pick = random.randint(0, len(dataset)-1)
        while pick in picks:
            pick = random.randint(0, len(dataset)-1)
        picks.append(pick)
    
    df = pd.DataFrame(dataset[picks])
    for column, typ in dataset.features.items():
        if isinstance(typ, datasets.ClassLabel):
            df[column] = df[column].transform(lambda i: typ.names[i])
    display(HTML(df.to_html()))

In [8]:
show_random_elements(dataset["train"])

,label,text
0,4 stars,"This place is really small, but has most of the basic farmer's market fruits and veggies that you expect, fresh and locally grown. The best thing here is the prices. For example, I got gigantic red bell peppers for 99 cents each. Compare this to the 2.50 charged nearby at Harris Teeter for red bells that are smaller and less fresh. This is the kind of place that I wish there were more of around Charlotte. It is well worth making a stop here when you are in the area."
1,2 star,"Dropped in today because I was in the mood for a Gyro and wanted to try someplace different than my usual Nero's stop. \n\nI will not be returning. \n\nThe gyro was overpriced, not particularly flavorful, and the fries were... just fries. I could get them from anywhere. Not what I'd expect from a meal running $12 with drink. \n\nThe entire time I was in there I was the only customer. Now I know why."
2,4 stars,"This place's signage has been up for a while and I've been waiting for them to open. They finally opened up today (1/2/10). Like other frozen yogurt places, it's a self serve set-up where you get your yogurt and add on the toppings you want. The flavors are pretty standard, but everything I sampled was delicious, especially the cheesecake. All the toppings looked extremely fresh, as is to be expected since it was their first day open. But the owner said they plan to use only fresh fruit and not the frozen stuff. It's 34 cents an ounce, which is pretty much on par with other places around town. The interior is brightly colored -- pinks and blues. They use picnic tables for seating, which is a cool idea. They also have kid-sized furniture for the little ones. Free wifi is also available. The owners are super nice. It's great to see some locally-owned businesses popping up in Mountain's Edge. I'm sure I'll be stopping by regularly to support them."
3,2 star,Very poor quality - food tasted reheated and old. When they first opened it was TOP Notch considering no one spoke English :-( but they were generous & thankful for our patronage. Once they obtained there liquor license all went down hill. It's appears to be reheated & reheated over and over again - and maybe even recycled dishes from leftovers. Idk - but nothing tastes fresh anymore
4,4 stars,"Excellent broccoli cheese soup! Recommend the salmon sandwich too. The salad bar had a lot of different items on it, but was missing some of the standards, i.e. black olives. But still worth the trip and I would go back here again."
5,4 stars,"Just two blocks away whenever I want a samich. Every sandwich that I have had has been delicious from PDC and I like how there is a pickle barrel. Low key laid back atmosphere with beers to drink and tvs up, or you can take out. I think the price is just a little hefty ($8-$10) but I'm okay with it due to the price and size. Service has been allright - nothing in particular to rave or boo about, but this place is a staple."
6,5 stars,"This is a great place. We're not very familiar with Korean food and needed recommendations from our server. We had the meatballs, chicken wings, and a pork dish. They were all full of flavor and not overly spicy as we had requested We also had the tempura fried ice cream and again it was great. The servings are generous and perfect for sharing. Highly recommend this restaurant ."
7,3 stars,"The food here tastes like food you could get at Fridays or Chilis or any American chain restaurant but for double the price .. I got the crabcake sandwich or burger or something, and thought it was kind of dry and over cooked. The buffalo wings were delicious, but you really can't go wrong with it since it just tastes like buffalo sauce .. \n\nI ordered the almond frozen hot chocolate to go, but the waiter brought it to me in the bowl thing. While it was admittedly beautiful served as it was, you realize that it only fills up like 3/4 of a tall starbucks cup when they transfer it and take off the huge amount of whipped cream! Fo

## 预处理数据

下载数据集到本地后，使用 Tokenizer 来处理文本，对于长度不等的输入数据，可以使用填充（padding）和截断（truncation）策略来处理。

Datasets 的 `map` 方法，支持一次性在整个数据集上应用预处理函数。

下面使用填充到最大长度的策略，处理整个数据集：

In [9]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")


def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)


tokenized_datasets = dataset.map(tokenize_function, batched=True)

/home/tiger/miniforge3/envs/py310/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [10]:
show_random_elements(tokenized_datasets["train"], num_examples=1)

,label,text,input_ids,token_type_ids,attention_mask
0,2 star,"My girlfriends and I are HUGE foodies and huge Tom Colicchio fans. My issue was that the three of us came to restaurant ready to spend money on a fantastic meal and the entire meal felt rushed. Our server made it clear that the kitchen was super \""slammed' and made us feel as though we didn't have the luxury of time to enjoy the various courses of our meal. He almost audibly sighed when we decided to order extra oysters and one of the servers had a conniption fit in front of us when he came to the table and saw that we were still eating our apps. To top it all off - we decided that we would do a \""taste test\"" between the Kobe Rib Eye and the \""regular\"" filet. It was very disappointing to say that while both steaks were good there was no wow factor for either. Then we discover that our original server left without introducing the new server. It was very obvious that the staff were ready for us to leave and this was disappointing since we spent quite a bit of money there and weren't really made to feel welcome. We left not talking about the great food - but about the disappointment we felt.","[101, 1422, 6124, 1116, 1105, 146, 1132, 145, 2591, 16523, 2094, 1905, 1105, 3321, 2545, 9518, 1596, 4313, 1186, 3899, 119, 1422, 2486, 1108, 1115, 1103, 1210, 1104, 1366, 1338, 1106, 4382, 2407, 1106, 4511, 1948, 1113, 170, 14820, 7696, 1105, 1103, 2072, 7696, 1464, 6169, 119, 3458, 9770, 1189, 1122, 2330, 1115, 1103, 3119, 1108, 7688, 165, 107, 7040, 112, 1105, 1189, 1366, 1631, 1112, 1463, 1195, 1238, 112, 189, 1138, 1103, 9886, 1104, 1159, 1106, 5548, 1103, 1672, 4770, 1104, 1412, 7696, 119, 1124, 1593, 12686, 3309, 4999, 4756, 1165, 1195, 1879, 1106, 1546, 3908, 184, 21878, 1116, ...]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...]","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...]"


### 数据抽样

使用 1000 个数据样本，在 BERT 上演示小规模训练（基于 Pytorch Trainer）

`shuffle()`函数会随机重新排列列的值。如果您希望对用于洗牌数据集的算法有更多控制，可以在此函数中指定generator参数来使用不同的numpy.random.Generator。

In [11]:
small_train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(1000))
small_eval_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(1000))
len(small_eval_dataset)

1000

## 微调训练配置

### 加载 BERT 模型

警告通知我们正在丢弃一些权重（`vocab_transform` 和 `vocab_layer_norm` 层），并随机初始化其他一些权重（`pre_classifier` 和 `classifier` 层）。在微调模型情况下是绝对正常的，因为我们正在删除用于预训练模型的掩码语言建模任务的头部，并用一个新的头部替换它，对于这个新头部，我们没有预训练的权重，所以库会警告我们在用它进行推理之前应该对这个模型进行微调，而这正是我们要做的事情。

In [12]:
from tqdm import tqdm
import time

# 降级后，这个交互式版本现在应该可以完美工作了
for i in tqdm(range(10), desc="测试进度条"):
    time.sleep(0.5)

测试进度条: 100%|██████████| 10/10 [00:05<00:00,  2.00it/s]


In [13]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("bert-base-cased", num_labels=5)
model

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

### 训练超参数（TrainingArguments）

完整配置参数与默认值：https://huggingface.co/docs/transformers/v4.36.1/en/main_classes/trainer#transformers.TrainingArguments

源代码定义：https://github.com/huggingface/transformers/blob/v4.36.1/src/transformers/training_args.py#L161

**最重要配置：模型权重保存路径(output_dir)**

In [14]:
from transformers import TrainingArguments

model_dir = "../models/bert-base-cased-finetune-yelp"

# logging_steps 默认值为500，根据我们的训练数据和步长，将其设置为100
training_args = TrainingArguments(output_dir=model_dir,
                                  per_device_train_batch_size=32,
                                  num_train_epochs=5,
                                  logging_steps=100)

In [15]:
# 完整的超参数配置
print(training_args)

TrainingArguments(
_n_gpu=1,
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
dispatch_batches=None,
do_eval=False,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_steps=None,
evaluation_strategy=no,
fp16=False,
fp16_backend=auto,
fp16_full_eval=False,
fp16_opt_level=O1,
fsdp=[],
fsdp_config={'min_num_params': 0, 'xla': False, 'xla_fsdp_grad_ckpt': False},
fsdp_min_num_params=0,
fsdp_transformer_layer_cls_to_wrap=None,
full_determinism=False,
gradient_accumulation_steps=1,
gradient_checkpointing=False,
gradient_checkpointing_kwargs=None,
greater_is_better=None,
group_by_le

### 训练过程中的指标评估（Evaluate)

**[Hugging Face Evaluate 库](https://huggingface.co/docs/evaluate/index)** 支持使用一行代码，获得数十种不同领域（自然语言处理、计算机视觉、强化学习等）的评估方法。 当前支持 **完整评估指标：https://huggingface.co/evaluate-metric**

训练器（Trainer）在训练过程中不会自动评估模型性能。因此，我们需要向训练器传递一个函数来计算和报告指标。 

Evaluate库提供了一个简单的准确率函数，您可以使用`evaluate.load`函数加载

In [16]:
import numpy as np
import evaluate

metric = evaluate.load("accuracy")


接着，调用 `compute` 函数来计算预测的准确率。

在将预测传递给 compute 函数之前，我们需要将 logits 转换为预测值（**所有Transformers 模型都返回 logits**）。

In [17]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

#### 训练过程指标监控

通常，为了监控训练过程中的评估指标变化，我们可以在`TrainingArguments`指定`evaluation_strategy`参数，以便在 epoch 结束时报告评估指标。

In [3]:
from transformers import TrainingArguments, Trainer
import os
training_args = TrainingArguments(output_dir=model_dir,
                                  evaluation_strategy="epoch", 
                                  per_device_train_batch_size=32,
                                  num_train_epochs=3,
                                  report_to=["tensorboard"],
                                  logging_dir=os.path.join(model_dir, "tb"),
                                  logging_steps=30)

NameError: name 'model_dir' is not defined

## 开始训练

### 实例化训练器（Trainer）

`kernel version` 版本问题：暂不影响本示例代码运行

In [2]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    compute_metrics=compute_metrics,
)
smaill_trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_eval_dataset,
    compute_metrics=compute_metrics,
)

NameError: name 'Trainer' is not defined

## 使用 nvidia-smi 查看 GPU 使用

为了实时查看GPU使用情况，可以使用 `watch` 指令实现轮询：`watch -n 1 nvidia-smi`:

```shell
Every 1.0s: nvidia-smi                                                   Wed Dec 20 14:37:41 2023

Wed Dec 20 14:37:41 2023
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.129.03             Driver Version: 535.129.03   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla T4                       Off | 00000000:00:0D.0 Off |                    0 |
| N/A   64C    P0              69W /  70W |   6665MiB / 15360MiB |     98%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+----------------------+

+---------------------------------------------------------------------------------------+
| Processes:                                                                            |
|  GPU   GI   CI        PID   Type   Process name                            GPU Memory |
|        ID   ID                                                             Usage      |
|=======================================================================================|
|    0   N/A  N/A     18395      C   /root/miniconda3/bin/python                6660MiB |
+---------------------------------------------------------------------------------------+
```
```shell
Wed Mar 18 00:02:42 2026
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.261.03             Driver Version: 535.261.03   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA A10                     On  | 00000000:65:01.0 Off |                  Off |
|  0%   62C    P0             155W / 150W |  22010MiB / 24564MiB |    100%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+----------------------+

+---------------------------------------------------------------------------------------+
| Processes:                                                                            |
|  GPU   GI   CI        PID   Type   Process name                            GPU Memory |
|        ID   ID                                                             Usage      |
|=======================================================================================|
|    0   N/A  N/A   2093349      C   ...er/miniforge3/envs/py310/bin/python    22002MiB |
+---------------------------------------------------------------------------------------+
```


In [21]:
smaill_trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,1.537900,1.258639,0.502000
2,1.104300,1.037631,0.545000
3,0.823600,0.969629,0.582000


TrainOutput(global_step=96, training_loss=1.12844114502271, metrics={'train_runtime': 136.6352, 'train_samples_per_second': 21.956, 'train_steps_per_second': 0.703, 'total_flos': 789354427392000.0, 'train_loss': 1.12844114502271, 'epoch': 3.0})

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss


In [19]:
small_test_dataset = tokenized_datasets["test"].shuffle(seed=64).select(range(100))

In [ ]:
trainer.evaluate(tokenized_datasets["test"])

In [1]:
#全量数据，抽样部分来验证（相对于1000，准确率从 0.54->0.65）
smaill_trainer.evaluate(small_test_dataset)

NameError: name 'smaill_trainer' is not defined

In [ ]:
smaill_trainer.evaluate(small_test_dataset)

In [20]:
trainer.evaluate(small_test_dataset)

{'eval_loss': 1.0822978019714355,
 'eval_accuracy': 0.54,
 'eval_runtime': 1.2797,
 'eval_samples_per_second': 78.142,
 'eval_steps_per_second': 10.158,
 'epoch': 3.0}

### 保存模型和训练状态

- 使用 `trainer.save_model` 方法保存模型，后续可以通过 from_pretrained() 方法重新加载
- 使用 `trainer.save_state` 方法保存训练状态

In [31]:
trainer.save_model(model_dir)

In [32]:
trainer.save_state()

In [ ]:
# trainer.model.save_pretrained("./")

## Homework: 使用完整的 YelpReviewFull 数据集训练，看 Acc 最高能到多少

In [ ]:
全量数据，抽样部分来验证（相对于1000，准确率从 0.54->0.65）